<a href="https://colab.research.google.com/github/jaswanthbhavanam-03/SivaSaiJaswanthBhavanam_INFO5731_Spring2026/blob/main/Bhavanam_Jaswanth_Assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [1]:
# Your code here
!pip -q install pandas requests tqdm nltk spacy stanza beautifulsoup4 lxml

import re
import time
import math
import requests
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# For Q3 (POS/NER/Dependency)
import spacy

# For constituency parsing + dependency parsing trees
import stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 7.4 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [2]:
# spaCy small English model
!python -m spacy download en_core_web_sm -q

# stanza English models (includes constituency + dependency)
stanza.download('en')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

API collection (10,000 papers)

In [6]:
import time
import requests
import pandas as pd
from tqdm import tqdm

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
FIELDS = "paperId,title,year,venue,abstract,url"

# Use 1 query or multiple queries to increase coverage
QUERIES = ["machine learning", "artificial intelligence", "data science", "information extraction"]

TARGET_ABSTRACTS = 10000
BATCH_SIZE = 100

headers = {
    # If you have an API key, uncomment:
    # "x-api-key": "YOUR_API_KEY"
}

def is_good_abstract(a):
    return isinstance(a, str) and len(a.strip()) > 30  # avoids empty/too-short

all_rows = []

for query in QUERIES:
    token = None
    print(f"\n🔎 Collecting for query: {query}")

    with tqdm(total=TARGET_ABSTRACTS, desc=f"Abstracts collected", leave=True) as pbar:
        # initialize progress bar with current count
        pbar.n = sum(is_good_abstract(r.get("abstract")) for r in all_rows)
        pbar.refresh()

        while sum(is_good_abstract(r.get("abstract")) for r in all_rows) < TARGET_ABSTRACTS:
            params = {"query": query, "fields": FIELDS, "limit": BATCH_SIZE}
            if token:
                params["token"] = token

            r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
            if r.status_code != 200:
                print("❌ Error:", r.status_code, r.text[:200])
                break

            data = r.json()
            papers = data.get("data", [])
            token = data.get("token")

            if not papers:
                print("No more results for this query.")
                break

            # Add papers
            before = sum(is_good_abstract(x.get("abstract")) for x in all_rows)
            for p in papers:
                all_rows.append({
                    "paperId": p.get("paperId"),
                    "title": p.get("title"),
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "abstract": p.get("abstract"),
                    "url": p.get("url"),
                    "query_used": query
                })
            after = sum(is_good_abstract(x.get("abstract")) for x in all_rows)
            pbar.update(after - before)

            time.sleep(0.2)

            if token is None:
                break

    # Stop early if we already hit target
    if sum(is_good_abstract(r.get("abstract")) for r in all_rows) >= TARGET_ABSTRACTS:
        break

# Create df and keep only rows with real abstracts
df = pd.DataFrame(all_rows)
df = df[df["abstract"].apply(is_good_abstract)].drop_duplicates(subset=["paperId"]).head(TARGET_ABSTRACTS).reset_index(drop=True)

print("\n✅ Final abstracts:", len(df))
df.head()


🔎 Collecting for query: machine learning


Abstracts collected:  51%|█████▏    | 5148/10000 [00:16<00:15, 312.68it/s]



🔎 Collecting for query: artificial intelligence


Abstracts collected:  71%|███████   | 7080/10000 [00:07<00:02, 1000.31it/s]



🔎 Collecting for query: data science


Abstracts collected:  71%|███████   | 7080/10000 [00:00<00:00, 10945.28it/s] 


❌ Error: 500 {"message": "Internal Server Error"}


🔎 Collecting for query: information extraction


Abstracts collected:  71%|███████   | 7080/10000 [00:00<00:00, 9126.16it/s]  

❌ Error: 500 {"message": "Internal Server Error"}


✅ Final abstracts: 6653


,paperId,title,year,venue,abstract,url,query_used
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,2024.0,Energies,In the era of burgeoning electric vehicle (EV)...,https://www.semanticscholar.org/paper/00000c33...,machine learning
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,2021.0,Journal of Medical Internet Research,Background Meditation apps have surged in popu...,https://www.semanticscholar.org/paper/0000238f...,machine learning
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,2023.0,PeerJ Computer Science,Background Humans must be able to cope with th...,https://www.semanticscholar.org/paper/00003156...,machine learning
3,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,2023.0,International Conference on Communication and ...,The growing number of cloud-based services has...,https://www.semanticscholar.org/paper/00005d68...,machine learning
4,00005f1b7e976068ca4b5a1b546d9945158b3bfc,Diffusion Generative Models for Designing Effi...,2024.0,Journal of Physical Chemistry A,"Diffusion generative models, a class of machin...",https://www.semanticscholar.org/paper/00005f1b...,machine learning


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
# Write code for each of the sub parts with proper comments.



# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
# Your code here



# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.